# Routing Workflows

Different requests need different handling. **Routing** categorizes the input first, then sends it to **one** specialized pipeline — instead of a one-size-fits-all prompt.

Example: a video-script tool. "Python functions" wants **educational** content; "surfing" wants **entertainment**. A single generic prompt can't nail both.

**Two steps:**
1. **Categorize** — one Claude call classifies the input into a predefined genre.
2. **Specialized processing** — use that category to pick the matching prompt template and generate.

> Illustrative, runnable notebook (course discusses the pattern, no downloadable — but quotes the categorize prompt, used here). Runs on Haiku 4.5.

## Setup

In [1]:
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())

from anthropic import Anthropic

client = Anthropic()
model = "claude-haiku-4-5-20251001"


def get_text(message):
    for block in message.content:
        if block.type == "text":
            return block.text
    return ""


def chat(prompt, max_tokens=500):
    return get_text(client.messages.create(
        model=model, max_tokens=max_tokens, temperature=1.0,
        messages=[{"role": "user", "content": prompt}],
    ))


print(f"Ready. Model: {model}")

Ready. Model: claude-haiku-4-5-20251001


## Categories + specialized templates

Each genre gets its own prompt template (only one will run per request, so each can be highly tuned).

In [2]:
TEMPLATES = {
    "Educational": "Write a ~120-word short-video script that turns the topic into a clear, engaging explanation using relatable examples and a thought-provoking question.",
    "Entertainment": "Write a ~120-word high-energy short-video script with trendy, culturally-relevant language and punchy visual hooks.",
    "Comedy": "Write a ~120-word short-video script that's sharp and funny, with an unexpected angle and good comedic timing.",
    "Personal vlog": "Write a ~120-word authentic, intimate first-person vlog script with conversational storytelling.",
    "Reviews": "Write a ~120-word decisive review-style script highlighting concrete strengths and weaknesses from experience.",
    "Storytelling": "Write a ~120-word immersive short-video script using vivid detail and an emotional arc.",
}

CATEGORIES = list(TEMPLATES.keys())
print(CATEGORIES)

['Educational', 'Entertainment', 'Comedy', 'Personal vlog', 'Reviews', 'Storytelling']


## The router — categorize the topic

The course's categorize prompt, constrained to return just the category name. We normalize the reply against the known list (with a safe fallback).

In [3]:
def route(topic):
    cats = "\n".join(f"- {c}" for c in CATEGORIES)
    prompt = f"""
Categorize the topic of a video into one of the listed categories.
Respond with ONLY the category name, exactly as written, nothing else.

<topic>{topic}</topic>

<categories>
{cats}
</categories>
"""
    raw = chat(prompt, max_tokens=20).strip()
    for c in CATEGORIES:
        if c.lower() in raw.lower():
            return c
    return "Educational"  # fallback if unrecognized


for t in ["Python functions", "surfing", "reviewing the new iPhone", "my morning routine"]:
    print(f"{t!r:35} -> {route(t)}")

'Python functions'                  -> Educational
'surfing'                           -> Entertainment
'reviewing the new iPhone'          -> Reviews
'my morning routine'                -> Personal vlog


## Dispatch — generate with the chosen template

Only the selected pipeline runs. Route → pick template → generate.

In [4]:
def generate_script(topic):
    category = route(topic)
    prompt = f"{TEMPLATES[category]}\n\nTopic: {topic}"
    script = chat(prompt, max_tokens=400)
    return category, script


for topic in ["Python functions", "surfing"]:
    category, script = generate_script(topic)
    print(f"===== {topic}  ->  routed to: {category} =====")
    print(script)
    print()

===== Python functions  ->  routed to: Educational =====
# Python Functions: Your Code's Superpowers

**[HOOK]**
Ever noticed how your phone has different apps for different tasks? Python functions work the same way.

**[EXPLANATION]**
A function is a reusable block of code that does one specific job. Instead of writing the same code repeatedly, you write it once and call it whenever needed.

**[RELATABLE EXAMPLE]**
Think of a coffee machine. You press a button (call the function), it grinds beans, adds water, and brews—all automatically. You don't remake those steps; the machine handles it.

In Python:
```
def make_coffee(bean_type):
    print(f"Brewing {bean_type}...")
```

**[THOUGHT-PROVOKING QUESTION]**
If you could automate repetitive tasks in your daily life, what would you choose?

That's exactly what functions do for code.

===== surfing  ->  routed to: Entertainment =====
# CATCH THE WAVE 🌊

[QUICK CUTS - energetic music]

**SPEAKER (hyped, fast-paced):**

"Yo, surfing isn't 

## Why routing works

- **Specialization** — input hits **one** optimized pipeline, not a generic prompt straining to cover every genre.
- **Independent tuning** — improve the educational template without touching entertainment.
- **Clean extension** — add a genre = add a template + a category; the router prompt just lists one more.

**When to use it:** diverse request types needing different approaches, categories you can define clearly, and a classification step Claude handles reliably — customer-service bots, content tools, anything where the right response depends on the *kind* of request.

**Compose it:** the router is just step 1 of a chain; each pipeline can itself be a chain, a parallelization, or an evaluator-optimizer loop.

Next: **Agents and tools** — where Claude, not your code, drives the control flow.